# SentinelGraph Member 2: Transaction AML Risk Baseline

This notebook validates the real processed AMLSim files before training. It never creates substitute data, changes split assignments, or trains when required inputs are missing. Run cells in order.

## 1. Install Dependencies

Colab runtimes commonly include most packages; this cell installs the project requirements. The supported project runtime is Python 3.10–3.13.

In [ ]:
%pip install -q 'numpy>=1.26,<3.0' 'pandas>=2.2,<3.0' 'pyarrow>=18,<22' 'scikit-learn>=1.5,<2.0' 'xgboost>=2.1.4,<3.0' 'joblib>=1.4,<2.0'
import sys
print('Python:', sys.version)

## 2. Select Dataset Source

For Colab ZIP upload, set `UPLOAD_DATASET_ZIP = True` and choose the archive. Alternatively mount Google Drive and set `DATA_DIR` to the dataset folder. In local Jupyter, use the project path or an explicit dataset directory.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import zipfile

UPLOAD_DATASET_ZIP = False
USE_GOOGLE_DRIVE = False
PROJECT_ROOT_SETTING = os.environ.get('SENTINELGRAPH_PROJECT_ROOT')
PROJECT_ROOT = Path(PROJECT_ROOT_SETTING or Path.cwd()).expanduser()
if not PROJECT_ROOT_SETTING and not (PROJECT_ROOT / 'src' / 'sentinelgraph').is_dir():
    for candidate in (PROJECT_ROOT / 'SentinelGraph_Member2', PROJECT_ROOT.parent):
        if (candidate / 'src' / 'sentinelgraph').is_dir():
            PROJECT_ROOT = candidate
            break
DATA_DIR = Path(os.environ.get('SENTINELGRAPH_DATA_DIR', PROJECT_ROOT / 'data' / 'processed' / 'amlsim_1m')).expanduser()
UPLOAD_ROOT = Path('/content/sentinelgraph_amlsim_upload')

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError('Google Drive mounting is available only in Google Colab.') from exc
    drive.mount('/content/drive')
    PROJECT_ROOT = Path(os.environ.get('SENTINELGRAPH_PROJECT_ROOT', '/content/drive/MyDrive/SentinelGraph_Member2')).expanduser()
    DATA_DIR = Path(os.environ.get('SENTINELGRAPH_DATA_DIR', '/content/drive/MyDrive/SentinelGraph/data/processed/amlsim_1m')).expanduser()

if UPLOAD_DATASET_ZIP:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError('Browser ZIP upload is available only in Google Colab.') from exc
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1:
        raise RuntimeError(f'Upload exactly one dataset ZIP; received {len(zip_names)} ZIP files.')
    archive_path = Path('/content') / zip_names[0]
    UPLOAD_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        root = UPLOAD_ROOT.resolve()
        for member in archive.infolist():
            destination = (UPLOAD_ROOT / member.filename).resolve()
            if destination != root and root not in destination.parents:
                raise RuntimeError(f'Unsafe path in ZIP archive: {member.filename}')
        archive.extractall(UPLOAD_ROOT)
    DATA_DIR = UPLOAD_ROOT

print('Project source root:', PROJECT_ROOT.resolve())
print('Dataset search root:', DATA_DIR.resolve())

## 3. Locate and Load the Real Files

All six expected files must be present together. The notebook stops here with a specific missing-file message if the dataset has not arrived.

In [ ]:
import pandas as pd

REQUIRED_FILES = (
    'transactions.parquet', 'labels.parquet', 'splits.parquet',
    'accounts.parquet', 'manifest.json', 'split_config.json',
)

def find_dataset_directory(search_root: Path) -> Path:
    if search_root.is_dir():
        candidates = [search_root, *[path.parent for path in search_root.rglob('manifest.json')]]
        for candidate in candidates:
            if all((candidate / name).is_file() for name in REQUIRED_FILES):
                return candidate
    present = sorted({path.name for path in search_root.rglob('*') if path.is_file()}) if search_root.exists() else []
    missing = [name for name in REQUIRED_FILES if name not in present]
    raise FileNotFoundError(
        'Processed AMLSim dataset is unavailable or incomplete. Required files together: '
        + ', '.join(REQUIRED_FILES)
        + '. Missing by filename: ' + (', '.join(missing) if missing else 'files exist but are not together')
        + ' Upload the approved dataset ZIP or set DATA_DIR to its folder; do not create replacement data.'
    )

DATA_DIR = find_dataset_directory(Path(DATA_DIR))
print('Validated file directory:', DATA_DIR.resolve())
transactions = pd.read_parquet(DATA_DIR / 'transactions.parquet')
labels = pd.read_parquet(DATA_DIR / 'labels.parquet')
splits = pd.read_parquet(DATA_DIR / 'splits.parquet')
accounts = pd.read_parquet(DATA_DIR / 'accounts.parquet')
with (DATA_DIR / 'manifest.json').open(encoding='utf-8') as stream:
    manifest = json.load(stream)
with (DATA_DIR / 'split_config.json').open(encoding='utf-8') as stream:
    split_config = json.load(stream)

for name, frame in [('transactions', transactions), ('labels', labels), ('splits', splits), ('accounts', accounts)]:
    print(f'{name}: rows={len(frame):,}, columns={list(frame.columns)}')
print('Manifest keys:', list(manifest) if isinstance(manifest, dict) else type(manifest).__name__)
print('Split configuration:')
print(json.dumps(split_config, indent=2, sort_keys=True))

## 4. Validate Schemas, IDs, Labels, and Chronology

Joins use canonical `transaction_id` with one-to-one validation. The configured ranges and observed assignments must match train `0–139`, validation `140–169`, and test `170–199`; any mismatch raises an error. No random split is created.

In [ ]:
import numpy as np

EXPECTED_RANGES = {'train': (0, 139), 'validation': (140, 169), 'test': (170, 199)}
REQUIRED_COLUMNS = {
    'transactions': {'transaction_id', 'amount', 'timestamp_step'},
    'labels': {'transaction_id', 'is_fraud'},
    'splits': {'transaction_id', 'split'},
}

def require_columns(frame: pd.DataFrame, required: set[str], name: str) -> None:
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f'{name} is missing required columns: {missing}')

def validate_id_column(frame: pd.DataFrame, name: str) -> None:
    if frame['transaction_id'].isna().any():
        raise ValueError(f'{name} contains missing transaction_id values')
    duplicate_count = int(frame['transaction_id'].duplicated().sum())
    if duplicate_count:
        raise ValueError(f'{name} contains {duplicate_count} duplicate transaction_id values')

for name, frame in [('transactions', transactions), ('labels', labels), ('splits', splits)]:
    require_columns(frame, REQUIRED_COLUMNS[name], name)
    validate_id_column(frame, name)

transaction_ids = set(transactions['transaction_id'])
label_ids = set(labels['transaction_id'])
split_ids = set(splits['transaction_id'])
coverage = {
    'transactions_without_labels': len(transaction_ids - label_ids),
    'transactions_without_splits': len(transaction_ids - split_ids),
    'labels_without_transactions': len(label_ids - transaction_ids),
    'splits_without_transactions': len(split_ids - transaction_ids),
}
print('Join coverage:', coverage)
if any(coverage.values()):
    raise ValueError(f'Transaction/label/split ID coverage mismatch: {coverage}')

if labels['is_fraud'].isna().any() or not set(labels['is_fraud'].unique()).issubset({0, 1, False, True}):
    raise ValueError('is_fraud must be non-null binary values 0/1')
missing_by_table = {
    name: {str(column): int(count) for column, count in frame.isna().sum().items() if count}
    for name, frame in [('transactions', transactions), ('labels', labels), ('splits', splits), ('accounts', accounts)]
}
print('Missing values by table/column:', missing_by_table)
required_non_null = {
    'transactions': ['transaction_id', 'amount', 'timestamp_step'],
    'labels': ['transaction_id', 'is_fraud'],
    'splits': ['transaction_id', 'split'],
}
for table_name, columns in required_non_null.items():
    frame = {'transactions': transactions, 'labels': labels, 'splits': splits}[table_name]
    invalid_columns = [column for column in columns if frame[column].isna().any()]
    if invalid_columns:
        raise ValueError(f'{table_name} has missing required values in: {invalid_columns}')
for table_name, frame in [('transactions', transactions), ('labels', labels), ('splits', splits), ('accounts', accounts)]:
    numeric = frame.select_dtypes(include=[np.number])
    if numeric.shape[1] and not np.isfinite(numeric.to_numpy(dtype=float)).all():
        raise ValueError(f'{table_name} contains missing or non-finite numeric values')
unexpected_splits = sorted(set(splits['split'].unique()) - set(EXPECTED_RANGES))
if unexpected_splits:
    raise ValueError(f'Unexpected split labels: {unexpected_splits}')

for column in ('amount', 'timestamp_step'):
    transactions[column] = pd.to_numeric(transactions[column], errors='coerce')
    if transactions[column].isna().any() or not np.isfinite(transactions[column].to_numpy(dtype=float)).all():
        raise ValueError(f'transactions.{column} contains missing, non-numeric, or non-finite values')
if (transactions['amount'] < 0).any():
    raise ValueError('transactions.amount contains negative values')
steps = transactions['timestamp_step'].to_numpy(dtype=float)
if (steps < 0).any() or not np.equal(steps, np.floor(steps)).all():
    raise ValueError('timestamp_step must contain non-negative integers')

merged = transactions.merge(labels, on='transaction_id', how='outer', validate='one_to_one', indicator='_label_join')
if not (merged['_label_join'] == 'both').all():
    raise ValueError('Transaction-label join is not complete')
merged = merged.drop(columns='_label_join').merge(
    splits, on='transaction_id', how='outer', validate='one_to_one', indicator='_split_join'
)
if not (merged['_split_join'] == 'both').all():
    raise ValueError('Transaction-split join is not complete')
merged = merged.drop(columns='_split_join')
if len(merged) != len(transactions) or merged['transaction_id'].duplicated().any():
    raise ValueError('Joined row count or transaction ID uniqueness changed')
print('Joined rows:', len(merged), '| IDs preserved:', merged['transaction_id'].equals(transactions['transaction_id']))

def extract_configured_ranges(config: object) -> dict[str, tuple[int, int]]:
    found: dict[str, tuple[int, int]] = {}
    start_names = {'start', 'start_step', 'min_step', 'min_timestamp_step', 'timestamp_step_start'}
    end_names = {'end', 'end_step', 'max_step', 'max_timestamp_step', 'timestamp_step_end'}
    def walk(value: object) -> None:
        if isinstance(value, dict):
            for split_name in EXPECTED_RANGES:
                candidate = value.get(split_name)
                if isinstance(candidate, (list, tuple)) and len(candidate) == 2:
                    found[split_name] = (int(candidate[0]), int(candidate[1]))
                elif isinstance(candidate, dict):
                    normalized = {str(key).lower(): item for key, item in candidate.items()}
                    start = next((normalized[key] for key in start_names if key in normalized), None)
                    end = next((normalized[key] for key in end_names if key in normalized), None)
                    if start is not None and end is not None:
                        found[split_name] = (int(start), int(end))
            for child in value.values():
                walk(child)
        elif isinstance(value, list):
            for child in value:
                walk(child)
    walk(config)
    return found

configured_ranges = extract_configured_ranges(split_config)
if configured_ranges != EXPECTED_RANGES:
    raise ValueError(
        f'split_config.json ranges {configured_ranges} do not exactly match required ranges {EXPECTED_RANGES}. '
        'Inspect the actual configuration format and resolve the conflict; do not override split assignments.'
    )

for split_name, (minimum, maximum) in EXPECTED_RANGES.items():
    subset = merged.loc[merged['split'] == split_name, 'timestamp_step']
    if subset.empty or int(subset.min()) != minimum or int(subset.max()) != maximum:
        observed = None if subset.empty else (int(subset.min()), int(subset.max()))
        raise ValueError(f'{split_name} observed step bounds {observed}; expected {(minimum, maximum)}')
    invalid = ~subset.between(minimum, maximum)
    if invalid.any():
        raise ValueError(f'{split_name} contains timestamp_step values outside {minimum}..{maximum}')

print('Validated chronological ranges:', configured_ranges)
print('Class counts by split:')
print(merged.groupby(['split', 'is_fraud'], observed=False).size().unstack(fill_value=0))
print('Baseline feature missing values:', merged[['amount', 'timestamp_step']].isna().sum().to_dict())
print('tx_behavior_id present but excluded:', 'tx_behavior_id' in merged.columns)

## 5. Baseline Features and Member 3 Feature Availability

The one canonical baseline list is `FEATURE_NAMES = ('amount', 'timestamp_step')` in `xgboost_model.py`. IDs remain identifiers only. The standalone project does not synthesize graph or historical features. Same-timestamp history requires Member 3's documented batch-history rule before temporal integration.

In [ ]:
import sys
PROJECT_ROOT = Path(PROJECT_ROOT)
if not (PROJECT_ROOT / 'src' / 'sentinelgraph').is_dir() and (PROJECT_ROOT / 'SentinelGraph_Member2').is_dir():
    PROJECT_ROOT = PROJECT_ROOT / 'SentinelGraph_Member2'
if not (PROJECT_ROOT / 'src' / 'sentinelgraph' / 'models' / 'xgboost_model.py').is_file():
    raise FileNotFoundError(
        f'Model source not found under PROJECT_ROOT={PROJECT_ROOT}. '
        'Set SENTINELGRAPH_PROJECT_ROOT to the project directory or upload the project source.'
    )
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from sentinelgraph.models import FEATURE_NAMES, train_and_evaluate

graph_module = PROJECT_ROOT / 'src/sentinelgraph/features/graph_features.py'
temporal_module = PROJECT_ROOT / 'src/sentinelgraph/features/temporal_features.py'
GRAPH_FEATURES_AVAILABLE = graph_module.is_file()
TEMPORAL_FEATURES_AVAILABLE = temporal_module.is_file()
print('Feature order:', FEATURE_NAMES)
if not GRAPH_FEATURES_AVAILABLE:
    print('Model B skipped: Member 3 graph_features.py is absent.')
if not TEMPORAL_FEATURES_AVAILABLE:
    print('Model C skipped: Member 3 temporal_features.py is absent.')
print('Integration TODO: inspect Member 3 contracts, confirm no future/label leakage and same-timestamp batch semantics, then add identical train/inference adapters before enabling Models B/C.')

## 6. Train with Chronological Splits

This baseline uses only the existing train and validation assignments for fitting/early stopping and validation-threshold selection. The test rows are passed only for final evaluation. Class weight is derived from the train label counts. No training runs before all previous validation cells succeed.

In [ ]:
train_frame = merged.loc[merged['split'] == 'train', ['transaction_id', 'is_fraud', *FEATURE_NAMES]].copy()
validation_frame = merged.loc[merged['split'] == 'validation', ['transaction_id', 'is_fraud', *FEATURE_NAMES]].copy()
test_frame = merged.loc[merged['split'] == 'test', ['transaction_id', 'is_fraud', *FEATURE_NAMES]].copy()

dataset_identity = {
    'manifest': manifest,
    'transactions_rows': int(len(transactions)),
    'transaction_id_min': str(transactions['transaction_id'].min()),
    'transaction_id_max': str(transactions['transaction_id'].max()),
}
split_definitions = {name: {'start_step': bounds[0], 'end_step': bounds[1]} for name, bounds in EXPECTED_RANGES.items()}

training_result = train_and_evaluate(
    train_frame=train_frame,
    validation_frame=validation_frame,
    test_frame=test_frame,
    output_dir=PROJECT_ROOT / 'models',
    dataset_identity=dataset_identity,
    split_definitions=split_definitions,
)
print('Model artifact:', training_result['model_path'])
print('Metadata:', training_result['metadata_path'])
print('Evaluation report:', training_result['report_path'])
print('Validation-selected threshold:', training_result['metadata']['selected_validation_threshold'])
print('Held-out test metrics:', json.dumps(training_result['metadata']['evaluation_metrics']['test'], indent=2))

## 7. Prediction Smoke Example

Run after training successfully creates `models/xgb-v1.json`. The transaction ID below is an interface example only; the function preserves it and does not generate predictions unless the saved artifact exists.

In [ ]:
from sentinelgraph.models import predict_transactions

example_record = {
    'transaction_id': str(merged.iloc[0]['transaction_id']),
    'amount': float(merged.iloc[0]['amount']),
    'timestamp_step': int(merged.iloc[0]['timestamp_step']),
}
prediction = predict_transactions([example_record], model_path=PROJECT_ROOT / 'models/xgb-v1.json')
print(prediction)

## 8. Integration Handoff

The prediction interface accepts `transaction_id`, `amount`, and `timestamp_step`; it returns `transaction_id`, `risk_score`, and `model_version`. Confirm with the backend team whether its field is named `risk_score` or `model_score`, the expected ID serialization, deployment artifact path, model-loading lifecycle, batching/error behavior, and whether a thresholded decision is also required. This project does not alter Member 4's API.